In [ ]:
from google.cloud import bigquery

PROJECT_ID = "qwiklabs-gcp-00-c521a9ba0b6e"
DATASET_ID = "emergency_response"
RAW_TABLE  = "emergency_calls"
MODEL_NAME = "response_time_model"

client = bigquery.Client(project=PROJECT_ID)
print("Client ready.")

Client ready.


In [ ]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "US"
client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset `{DATASET_ID}` ready.")

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_uri(
    "gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv",
    f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}",
    job_config=job_config,
)
load_job.result()

table = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}")
print(f"Loaded {table.num_rows:,} rows into `{RAW_TABLE}`.")

Dataset `emergency_response` ready.
Loaded 50,000 rows into `emergency_calls`.


In [ ]:
# Previewing
preview = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}` LIMIT 10
""").to_dataframe()
preview

,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37
5,17582,2023-01-01 02:28:50+00:00,Rescue,Downtown,Snowy,Sunday,2,High,25.15,6,28.48
6,21624,2023-01-01 02:44:06+00:00,Rescue,Oakmont,Snowy,Sunday,2,High,3.95,9,19.30
7,36793,2023-01-01 02:53:54+00:00,Fire,Riverside,Sunny,Sunday,2,High,5.87,10,10.72
8,41350,2023-01-01 03:52:33+00:00,Police,Greenfield,Windy,Sunday,3,High,6.66,5,20.55
9,32092,2023-01-01 04:09:23+00:00,Police,Maplewood,Snowy,Sunday,4,High,15.50,13,22.98


In [ ]:
stats = client.query(f"""
    SELECT
        COUNT(*)                      AS row_count,
        ROUND(AVG(response_time), 2)  AS avg_response_time,
        ROUND(MIN(response_time), 2)  AS min_response_time,
        ROUND(MAX(response_time), 2)  AS max_response_time,
        ROUND(AVG(distance_to_station), 2) AS avg_distance,
        ROUND(AVG(units_available), 2)     AS avg_units
    FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
""").to_dataframe()
stats

,row_count,avg_response_time,min_response_time,max_response_time,avg_distance,avg_units
0,50000,17.45,2.01,36.55,15.08,8.0


In [ ]:
by_category = client.query(f"""
    SELECT
        call_type,
        traffic_level,
        COUNT(*)                     AS num_calls,
        ROUND(AVG(response_time), 2) AS avg_response_time
    FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
    GROUP BY call_type, traffic_level
    ORDER BY avg_response_time DESC
""").to_dataframe()
by_category

,call_type,traffic_level,num_calls,avg_response_time
0,Rescue,High,4154,20.13
1,Police,High,4237,20.12
2,Fire,High,4204,20.09
3,Medical,High,4149,20.04
4,Medical,Medium,4175,17.15
5,Police,Medium,4197,17.15
6,Fire,Medium,4193,17.13
7,Rescue,Medium,4090,17.12
8,Police,Low,4102,15.19
9,Rescue,Low,4157,15.07


In [ ]:
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`
OPTIONS(
    model_type      = 'LINEAR_REG',
    input_label_cols = ['response_time']
) AS
SELECT
    * EXCEPT(call_id, call_timestamp)
FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
"""

print("Training model — this may take a minute or two...")
client.query(create_model_sql).result()
print(f"Model `{MODEL_NAME}` created.")

Training model — this may take a minute or two...
Model `response_time_model` created.


In [ ]:
evaluation = client.query(f"""
    SELECT *
    FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`)
""").to_dataframe()
evaluation

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.761934,4.827846,0.015117,1.501825,0.831417,0.83146


In [ ]:
prediction = client.query(f"""
    SELECT
        predicted_response_time,
        call_type,
        location,
        weather_condition,
        day_of_week,
        traffic_level,
        time_of_day,
        distance_to_station,
        units_available
    FROM ML.PREDICT(
        MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`,
        (
            SELECT
                'Medical' AS call_type,
                'Downtown' AS location,
                'Snowy' AS weather_condition,
                'Monday' AS day_of_week,
                'High' AS traffic_level,
                18 AS time_of_day,
                12.5 AS distance_to_station,
                3 AS units_available
            UNION ALL
            SELECT
                'Fire', 'Greenfield', 'Sunny', 'Saturday', 'Low',
                9, 4.2, 12
        )
    )
""").to_dataframe()
prediction

,predicted_response_time,call_type,location,weather_condition,day_of_week,traffic_level,time_of_day,distance_to_station,units_available
0,23.417662,Medical,Downtown,Snowy,Monday,High,18,12.5,3
1,6.927584,Fire,Greenfield,Sunny,Saturday,Low,9,4.2,12
